# Fake News & Source Credibility Detector — Colab Training

**Before running:** Runtime → Change runtime type → **T4 GPU**

Steps:
1. Mount Drive & upload project
2. Install dependencies
3. Run Phase 1–2 (data + features)
4. Run Phase 4 (TF-IDF baseline)
5. **Run Phase 5 (DeBERTa fine-tuning) ← main training cell**
6. Download trained weights

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────
# Upload your project folder to Google Drive first, then mount it here.
# Alternatively, use the file upload button to upload a zip.
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# Option A — project already in Drive:
PROJECT_DRIVE = '/content/drive/MyDrive/FakeNews_ML'   # adjust path if needed
PROJECT_LOCAL = '/content/FakeNews_ML'

if os.path.exists(PROJECT_DRIVE):
    if not os.path.exists(PROJECT_LOCAL):
        shutil.copytree(PROJECT_DRIVE, PROJECT_LOCAL)
    print('Project copied to', PROJECT_LOCAL)
else:
    print(f'{PROJECT_DRIVE} not found — upload your project zip in the next cell.')

In [ ]:
# ── Cell 2b (alternative): Upload project zip ─────────────────────────
# Run this cell ONLY if you did not use Google Drive above.
from google.colab import files
import zipfile, os

uploaded = files.upload()   # pick FakeNews_ML.zip from your Mac
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')
PROJECT_LOCAL = '/content/FakeNews_ML'
print('Extracted to', PROJECT_LOCAL)

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────
# Colab already has torch/numpy; only install what's missing.
%pip install -q transformers datasets sentencepiece vaderSentiment \
              langdetect shap mlflow tqdm fastapi uvicorn joblib
import nltk
for pkg in ['stopwords', 'punkt', 'punkt_tab', 'opinion_lexicon']:
    nltk.download(pkg, quiet=True)
print('Dependencies installed.')

In [ ]:
# ── Cell 4: Run Phase 1–2 (data loading + feature engineering) ────────
# Skip this cell if you already have data/train.csv, data/val.csv, data/test.csv
# (i.e. you uploaded pre-built CSVs with your project zip).
os.chdir(PROJECT_LOCAL)
!python credibility_detector_phases123.py 2>&1 | tail -30

In [ ]:
# ── Cell 5: Verify data ────────────────────────────────────────────────
import pandas as pd
os.chdir(PROJECT_LOCAL)
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'data/{split}.csv')
    nan_n = df['credibility_score'].isna().sum()
    print(f'{split}: {len(df):,} rows  credibility_score NaN={nan_n}')

In [ ]:
# ── Cell 6: Run Phase 4 (TF-IDF baseline) ────────────────────────────
# Sets the benchmark MAE that DeBERTa must beat.
os.chdir(PROJECT_LOCAL)
!python phase4_baseline_1.py --no-shap

In [ ]:
# ── Cell 7: Train DeBERTa (Phase 5) — MAIN TRAINING ──────────────────
#
# Expected on T4 (~15 GB VRAM):
#   Epoch time : 8–12 minutes
#   Total (5ep): 40–60 minutes
#   val_MAE    : ~0.20–0.24  (baseline 0.2867)
#
os.chdir(PROJECT_LOCAL)
!python phase5_deberta.py --train --device cuda

In [ ]:
# ── Cell 8: Check results ─────────────────────────────────────────────
import json
r = json.loads(open(f'{PROJECT_LOCAL}/models/deberta_results.json').read())
print(f"Best val MAE  : {r['best_val_MAE']}")
print(f"Best epoch    : {r['best_epoch']}")
print(f"Test MAE      : {r['test']['MAE']}")
print(f"Test Pearson r: {r['test']['Pearson_r']}")
print(f"Test F1       : {r['test']['Macro_F1']}")
print(f"Baseline MAE  : {r['baseline_MAE']}")
print(f"Improvement   : {r['improvement']:+.4f}")

In [ ]:
# ── Cell 9: Save weights back to Drive ────────────────────────────────
# Run this so you don't lose the trained weights when Colab disconnects.
import shutil
dst = '/content/drive/MyDrive/FakeNews_ML/models'
os.makedirs(dst, exist_ok=True)
for f in ['deberta_best.pt', 'deberta_results.json', 'baseline_tfidf.pkl',
          'baseline_results.json']:
    src = f'{PROJECT_LOCAL}/models/{f}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Saved {f}')

tok_dst = dst + '/deberta_tokenizer'
if os.path.exists(f'{PROJECT_LOCAL}/models/deberta_tokenizer'):
    if os.path.exists(tok_dst):
        shutil.rmtree(tok_dst)
    shutil.copytree(f'{PROJECT_LOCAL}/models/deberta_tokenizer', tok_dst)
    print('Saved deberta_tokenizer/')
print('Done — weights saved to Google Drive.')

In [ ]:
# ── Cell 10 (optional): Download weights directly ─────────────────────
# Use this if you prefer to download to your Mac instead of Drive.
import shutil
from google.colab import files
shutil.make_archive('/content/trained_models', 'zip',
                    f'{PROJECT_LOCAL}/models')
files.download('/content/trained_models.zip')